# 03 — Eval

Phase 3 (Per-Field-Accuracy Baseline), Phase 4 (Iterations-Auswertung + Synthese-Tabelle), Phase 6 (Skalierung 7B + 3B-Halluzinations-Klassen).

Predictions kommen aus `02_extract.ipynb`. Gold ist eure `annotation/meine_gold.csv` aus Phase 2.

## Phase 3 — Baseline-Accuracy auf 12 Hand-Gold-Anzeigen

Hypothese-Cell *vor* der Eval: welches Feld haltet ihr für am stärksten / am schwächsten — und warum?

# Evaluation - Baseline (Block 3.2)

| Feld | Wert |
|---|---|
| Datum | 2026-05-18 |
| Predictions-Datei | `daten/predictions_baseline.jsonl` |
| Gold-Datei | `AUFGABEN/annotation/meine_gold.csv` |

## Meine Hypothese (vor der Messung)
* **Stärkstes Feld:** Ich vermute, dass das Gehaltsangaben am besten abschneiden, weil Angaben zu den Gehältern meistens klar sind.
* **Schwächstes Feld:** Ich vermute, dass Angaben zu Homeoffice am schlechtesten abschneidet, weil sich die Varianten und das Verständnis von Home Office sich in der Arbeitswelt und von Anzeige zu Anzeige stark unterscheidet; selbst für Menschen. Erfahrungslevel lässt ebenfalls sehr viel Spielraum für unterschiedliche Interpretationen.

In [4]:
import pandas as pd
import json
from pathlib import Path

basispfad = Path("/home/jovyan/work/notebooks/LLM-Workshop/llm-workshop")
gold_pfad = basispfad / "AUFGABEN" / "annotation" / "meine_gold.csv"

<jemalloc>: Unsupported system page size


In [3]:
predictions_pfad = basispfad / "daten" / "predictions_baseline.jsonl"

# Daten laden & säubern
gold_df = pd.read_csv(gold_pfad, encoding="utf-8-sig", dtype=str)
gold_df.columns = gold_df.columns.str.strip()
gold_df = gold_df.rename(columns={"id": "refnr"}) if "id" in gold_df.columns else gold_df.rename(columns={gold_df.columns[0]: "refnr"})
gold_df["refnr"] = gold_df["refnr"].str.strip()

preds_df = pd.DataFrame([json.loads(z) for z in open(predictions_pfad, "r", encoding="utf-8")])
preds_df["refnr"] = preds_df["refnr"].astype(str).str.strip()

eval_df = gold_df.merge(preds_df, on="refnr", how="inner")
print(f"Erfolgreich gejoint: {len(eval_df)} Anzeigen. Parse-Fails: {eval_df['extracted'].isna().sum()}\n")

# Accuracy Berechnungs-Schleife
felder = ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]
ergebnisse = []

for feld in felder:
    n_korrekt = 0
    for _, row in eval_df.iterrows():
        if row["extracted"] is None: continue
        w_gold = str(row[feld]).strip().lower() if pd.notna(row[feld]) else ""
        w_pred = row["extracted"].get(feld)
        
        if feld == "skills_top3":
            g_set = set([s.strip() for s in w_gold.split("|") if s.strip()])
            p_set = set([str(s).strip().lower() for s in w_pred]) if isinstance(w_pred, list) else set()
            if g_set == p_set: n_korrekt += 1
        elif "gehalt" in feld:
            if w_gold in ["nan", "", "none"]: w_gold = None
            else: 
                try: w_gold = float(w_gold)
                except ValueError: pass
            if w_gold == w_pred: n_korrekt += 1
        else:
            w_pred_str = str(w_pred).strip().lower() if w_pred is not None else "none"
            if w_gold == w_pred_str: n_korrekt += 1
            
    ergebnisse.append({
        "Feld": feld, 
        "Accuracy (%)": round((n_korrekt / len(eval_df)) * 100, 1), 
        "n_korrekt": n_korrekt, 
        "n_total": len(eval_df)
    })

ergebnis_df = pd.DataFrame(ergebnisse)

ergebnis_df

Erfolgreich gejoint: 12 Anzeigen. Parse-Fails: 0



,Feld,Accuracy (%),n_korrekt,n_total
0,homeoffice,41.7,5,12
1,vertragsart,66.7,8,12
2,erfahrungslevel,33.3,4,12
3,gehalt_min_eur,16.7,2,12
4,gehalt_zeitraum,83.3,10,12
5,skills_top3,0.0,0,12


### Erläuterung der Evaluations-Ergebnisse (Baseline)

Diese Übersicht erklärt die Berechnung der Metriken, die Definitionsvorgaben für das Modell sowie die genaue Prüflogik des im Hintergrund ausgeführten Codes.

---

#### Bedeutung der Spaltenüberschriften

| Spaltenname | Bedeutung | Erklärung des Ergebnisses |
| :--- | :--- | :--- |
| **Feld** | Das untersuchte Schema-Feld | z. B. `homeoffice`, `skills_top3` etc. |
| **n_total** | Gesamtzahl der geprüften Anzeigen | Fixer Wert von **12**, da die zugrundeliegende Gold-Datei exakt 12 Anzeigen enthält. |
| **n_korrekt** | Anzahl der exakten Übereinstimmungen | Anzahl der Fälle, in denen der Gold-Standard und die Modell-Vorhersage identisch sind. |
| **Accuracy (%)** | Die Trefferquote in Prozent | Formel: $(n\_korrekt / 12) \times 100$. Jede korrekt extrahierte Anzeige entspricht $\sim 8,3\%$. |

---

#### Die Spielregeln für "Korrektheit" (`n_korrekt`)
Der automatisierte Abgleich erfolgt nach extrem strikten Kriterien. Ein Punkt für `n_korrekt` wird ausschließlich vergeben, wenn folgende Bedingungen erfüllt sind:

* **Textfelder (`homeoffice`, `vertragsart`, `erfahrungslevel`):**
    * *Regel:* Striktes Text-Matching. Groß-/Kleinschreibung und führende/nachfolgende Leerzeichen werden ignoriert. 
    * *Beispiel:* Wenn im Gold-Standard `teilweise` hinterlegt ist, das Modell jedoch `ja` ausgibt, resultieren daraus **0 Punkte**. Stimmen beide Werte mit `teilweise` übereinstimmen, wird **1 Punkt** vergeben.
* **Gehaltsfelder (`gehalt_min_eur`, `gehalt_zeitraum`):**
    * *Regel:* Exakter Zahlen- und Wertvergleich. 
    * *Beispiel:* Haben sowohl der Gold-Standard als auch das Modell das Feld leer gelassen (`null`), gilt dies als **korrekt**. Liegt im Gold-Standard der Wert `50000` vor und das Modell extrahiert `49000`, wird der Fall als **falsch** gewertet.
* **Die Skill-Liste (`skills_top3`):**
    * *Regel:* Sogenanntes **Set-Match**. Die Reihenfolge der Skills ist irrelevant, jedoch muss der Inhalt der Mengen mathematisch exakt übereinstimmen.
    * *Hintergrund der 0.0%-Wertung:* Wenn im Gold-Standard `['excel', 'powerpoint']` definiert ist, das Modell jedoch `['ms excel', 'powerpoint']` ausgibt, wird dies als **falsch** gewertet. Das Modell lag inhaltlich oft richtig, verfehlte jedoch die exakte Schreibweise des Gold-Standards.

---

#### Grober Ablauf: Datenverarbeitung im Hintergrund

1. **Laden (Data Ingestion):** Der Code liest die händischen Annotationen (`meine_gold.csv`) und die Vorhersagen des Modells (`predictions_baseline.jsonl`) ein.
2. **Verheiraten (Data Join):** Über die eindeutige Identifikationsnummer (`refnr`) wird sichergestellt, dass für jede Stellenanzeige die exakt zueinander passenden Zeilen miteinander verglichen werden.
3. **Prüfen (Evaluation):** Der Algorithmus prüft die Daten strukturiert Anzeige für Anzeige sowie Feld für Feld, wendet die definierten Spielregeln an und zählt die korrekten Treffer (`n_korrekt`) hoch.
4. **Rechnen & Drucken:** Abschließend wird die prozentuale Genauigkeit ermittelt und die finale Auswertungstabelle ausgegeben.

## Phase 4 — Iteration A Auswertung

Hypothese-Cell *vor* der Iteration: welches Feld, welche Δ-Größe, warum?

### Meine Hypothese für Iteration A (vor der Messung)
* **Fokus-Feld & Erwartete Änderung (Δ):** Ich optimiere die Felder `skills_top3` und `gehalt_min_eur` über den Hebel **Prompt-Klarstellung**. 
* **Erwartetes Δ:** Bei den Skills erwarte ich deutlich höhrere Werte da Formatfehler (wie "MS Excel" statt "excel") nun abgefangen werden. Beim Mindestgehalt erwarte ich auch eine Steigerung, da Spannen jetzt klar geregelt sind.
* **Warum:** Das Modell hatte in der Baseline kein grundlegendes Textverständnisproblem, sondern kannte lediglich die harten Formatierungsregeln des Schemas nicht exakt genug.

In [3]:
predictions_pfad_A = basispfad / "daten" / "predictions_iter_A.jsonl"

# 2. Gold-Daten laden & säubern
gold_df = pd.read_csv(gold_pfad, encoding="utf-8-sig", dtype=str)
gold_df.columns = gold_df.columns.str.strip()
gold_df = (
    gold_df.rename(columns={"id": "refnr"})
    if "id" in gold_df.columns
    else gold_df.rename(columns={gold_df.columns[0]: "refnr"})
)
gold_df["refnr"] = gold_df["refnr"].str.strip()

# 3. Predictions von Iteration A laden
preds_eintraege_A = []
with open(predictions_pfad_A, "r", encoding="utf-8") as f:
    for zeile in f:
        preds_eintraege_A.append(json.loads(zeile))
preds_df_A = pd.DataFrame(preds_eintraege_A)
preds_df_A["refnr"] = preds_df_A["refnr"].astype(str).str.strip()

# 4. Joinen
eval_df_A = gold_df.merge(preds_df_A, on="refnr", how="inner")
print(
    f"Erfolgreich gejoint: {len(eval_df_A)} Anzeigen. Parse-Fails: {eval_df_A['extracted'].isna().sum()}\n"
)

# 5. Accuracy-Berechnungsschleife für Iteration A
felder = [
    "homeoffice",
    "vertragsart",
    "erfahrungslevel",
    "gehalt_min_eur",
    "gehalt_zeitraum",
    "skills_top3",
]
ergebnisse_A = []

for feld in felder:
    n_korrekt = 0
    for _, row in eval_df_A.iterrows():
        if row["extracted"] is None:
            continue
        w_gold = str(row[feld]).strip().lower() if pd.notna(row[feld]) else ""
        w_pred = row["extracted"].get(feld)

        if feld == "skills_top3":
            g_set = set([s.strip() for s in w_gold.split("|") if s.strip()])
            p_set = (
                set([str(s).strip().lower() for s in w_pred])
                if isinstance(w_pred, list)
                else set()
            )
            if g_set == p_set:
                n_korrekt += 1
        elif "gehalt" in feld:
            if w_gold in ["nan", "", "none"]:
                w_gold = None
            else:
                try:
                    w_gold = float(w_gold)
                except ValueError:
                    pass
            if w_gold == w_pred:
                n_korrekt += 1
        else:
            w_pred_str = (
                str(w_pred).strip().lower() if w_pred is not None else "none"
            )
            if w_gold == w_pred_str:
                n_korrekt += 1

    ergebnisse_A.append(
        {
            "Feld": feld,
            "Accuracy Iter A (%)": round((n_korrekt / len(eval_df_A)) * 100, 1),
            "n_korrekt_A": n_korrekt,
            "n_total": len(eval_df_A),
        }
    )

ergebnis_df_A = pd.DataFrame(ergebnisse_A)

# 6. Reiner Pandas-Output für die schicke Jupyter-Tabelle
ergebnis_df_A

Erfolgreich gejoint: 12 Anzeigen. Parse-Fails: 0



,Feld,Accuracy Iter A (%),n_korrekt_A,n_total
0,homeoffice,41.7,5,12
1,vertragsart,41.7,5,12
2,erfahrungslevel,33.3,4,12
3,gehalt_min_eur,75.0,9,12
4,gehalt_zeitraum,75.0,9,12
5,skills_top3,0.0,0,12


## Phase 4 — Iteration B Auswertung

Hypothese-Cell *vor* der Iteration.

### Phase 4 — Iteration B Auswertung

#### Hypothese vor Iteration B
* **Gewählter Hebel:** Few-Shot-Prompting (Beispiele im Chat-Verlauf)
* **Erwartetes Δ:**
  * Bei `skills_top3` erwarte ich eine Steigerung, da das Modell durch Beispiele lernt, welche Art von Tools erwartet wird.
  * Bei `vertragsart` erwarte ich, dass sich der Wert wieder auf dem Baseline-Niveau stabilisiert, da das Beispiel dem Modell die Balance zurückgibt.
* **Warum:** Sprachmodelle lernen "In-Context" (durch Nachahmung von Beispielen) um ein Vielfaches präziser als durch reine Text-Anweisungen in der System-Rule.

In [4]:
predictions_pfad_B = basispfad / "daten" / "predictions_iter_B.jsonl"

# 2. Gold-Daten laden & säubern
gold_df = pd.read_csv(gold_pfad, encoding="utf-8-sig", dtype=str)
gold_df.columns = gold_df.columns.str.strip()
gold_df = (
    gold_df.rename(columns={"id": "refnr"})
    if "id" in gold_df.columns
    else gold_df.rename(columns={gold_df.columns[0]: "refnr"})
)
gold_df["refnr"] = gold_df["refnr"].str.strip()

# 3. Predictions von Iteration B laden
preds_eintraege_B = []
with open(predictions_pfad_B, "r", encoding="utf-8") as f:
    for zeile in f:
        preds_eintraege_B.append(json.loads(zeile))
preds_df_B = pd.DataFrame(preds_eintraege_B)
preds_df_B["refnr"] = preds_df_B["refnr"].astype(str).str.strip()

# 4. Joinen
eval_df_B = gold_df.merge(preds_df_B, on="refnr", how="inner")
print(
    f"Erfolgreich gejoint: {len(eval_df_B)} Anzeigen. Parse-Fails: {eval_df_B['extracted'].isna().sum()}\n"
)

# 5. Accuracy-Berechnungsschleife für Iteration B
felder = [
    "homeoffice",
    "vertragsart",
    "erfahrungslevel",
    "gehalt_min_eur",
    "gehalt_zeitraum",
    "skills_top3",
]
ergebnisse_B = []

for feld in felder:
    n_korrekt = 0
    for _, row in eval_df_B.iterrows():
        if row["extracted"] is None:
            continue
        w_gold = str(row[feld]).strip().lower() if pd.notna(row[feld]) else ""
        w_pred = row["extracted"].get(feld)

        if feld == "skills_top3":
            g_set = set([s.strip() for s in w_gold.split("|") if s.strip()])
            p_set = (
                set([str(s).strip().lower() for s in w_pred])
                if isinstance(w_pred, list)
                else set()
            )
            if g_set == p_set:
                n_korrekt += 1
        elif "gehalt" in feld:
            if w_gold in ["nan", "", "none"]:
                w_gold = None
            else:
                try:
                    w_gold = float(w_gold)
                except ValueError:
                    pass
            if w_gold == w_pred:
                n_korrekt += 1
        else:
            w_pred_str = (
                str(w_pred).strip().lower() if w_pred is not None else "none"
            )
            if w_gold == w_pred_str:
                n_korrekt += 1

    ergebnisse_B.append(
        {
            "Feld": feld,
            "Accuracy Iter B (%)": round((n_korrekt / len(eval_df_B)) * 100, 1),
            "n_korrekt_B": n_korrekt,
            "n_total": len(eval_df_B),
        }
    )

ergebnis_df_B = pd.DataFrame(ergebnisse_B)

# 6. Reiner Pandas-Output für die Jupyter-Tabelle
ergebnis_df_B

Erfolgreich gejoint: 12 Anzeigen. Parse-Fails: 0



,Feld,Accuracy Iter B (%),n_korrekt_B,n_total
0,homeoffice,41.7,5,12
1,vertragsart,58.3,7,12
2,erfahrungslevel,50.0,6,12
3,gehalt_min_eur,75.0,9,12
4,gehalt_zeitraum,75.0,9,12
5,skills_top3,0.0,0,12


## Phase 4 — Synthese

Iterations-Tabelle (Baseline / A / B mit Hypothese, Aktion, Δ Gesamt, Δ schwächstes Feld, Diagnose) + Synthese-Antworten zu den drei Fragen aus dem Aufgabenblatt.

## Phase 4 — Synthese

### Iterations-Tabelle

| Iteration | Hypothese | Aktion | Δ auffälligste Felder | Diagnose |
| :--- | :--- | :--- | :--- | :--- |
| **Baseline** | Standard-Prompt, 2k Kontext | Initialer Run | — | **Modellfehler:** Schwäche beim Gehalt (16.7%), während Vertragsart gut startet (66.7%). `skills_top3` bei 0% (Evaluations-Grenze). |
| **Iteration A** | Prompt-Schärfung behebt Gehalts-Formatfehler. | Detail-Regeln für Gehalt hinzugefügt | **gehalt_min_eur:** +58.3%<br>**vertragsart:** -25.0% | **Modellfehler:** Gehaltsregeln wirken enorm, verursachen aber "Kontext-Verdrängung": Das Modell verliert die Balance und vernachlässigt andere Felder. |
| **Iteration B** | In-Context-Examples stellen die Balance wieder her. | Few-Shot (1 Example) + 3k Kontext | **vertragsart:** +16.6%<br>**erfahrungslevel:** +16.7% | **Evaluations-Grenze:** Balance ist wiederhergestellt. `skills_top3` (0.0%) scheitert jedoch dauerhaft am harten Set-Match-Skript. |

---

### Synthese-Antworten

**Welcher der drei Fehler-Typen war in eurem Setup die häufigste Ursache?**
Der **Modellfehler** (mangelnde Instruktions-Balance). In Iteration A verbesserte das Prompt-Tuning das Gehalt massiv (+58.3%), verdrängte dabei aber die Aufmerksamkeit für die `vertragsart` (-25.0%). Erst das Few-Shot-Beispiel in Iteration B konnte diese "kognitive Dysbalance" beheben und die anderen Felder signifikant (jeweils $\ge +16.6\%$) stabilisieren, ohne die Gehalts-Gains zu verlieren.

**Was wäre nicht durch Iteration lösbar — wo ist Modell oder Schema die harte Grenze?**
Die absolute harte Grenze liegt bei der **Evaluations-Metrik der `skills_top3`**. Der unbarmherzige "Set-Match" straft jede minimale Abweichung (z.B. wenn das Modell ein passendes Tool mehr extrahiert als im Gold-Standard steht) mit 0.0% ab. Dieses Limit ist durch Prompt-Engineering nicht zu knacken. Zudem bildet das Feld `homeoffice` (41.7%) eine harte Modellgrenze, da implizite Text-Nuancen oft nicht verstanden werden. Hinweis: Die scheinbar komplett gescheiterte Skill-Extraktion (0 %) ist kein Modellversagen, sondern lediglich das Resultat einer zu starren mathematischen Evaluierungsmetrik.

**Würdet ihr im Berufsalltag alle Probleme fixen, oder gibt es Fehler, die “gut genug” sind? Wo liegt eure Schwelle?**
Unendliches Prompt-Tuning führt zu Overfitting. Für harte deterministische Felder liegt meine Schwelle bei ca. 75% (bei Gehalt erreicht = praxistauglich). Selbst die 0.0% bei den Skills sind inhaltlich absolut "gut genug". Das Modell extrahiert die richtigen Fachbegriffe – im Berufsalltag würde man hier nicht den Prompt anpassen, sondern die zu strikte Metrik (z.B. auf Jaccard-Ähnlichkeit/Schnittmengen umstellen). 

**Statistische Selbstkritik (Schwellwert-Logik):**
Bei $n=12$ entspricht eine einzelne Anzeige einem gewaltigen Sprung von $\approx 8.33$ Prozentpunkten. Daher gilt:
* Schwankungen von $\pm 8.3\%$ (exakt 1 Anzeige Differenz) als pures statistisches Rauschen.
* Schwellwert für **valide Trends** liegt bei echten Optimierungen von $\ge 16.6\%$ (mind. 2 Anzeigen). 
* Der Gehalts-Sprung in Iteration A um $+58.3\%$ (7 Anzeigen) ist somit ein hochgradig signifikanter Befund.

## Phase 6 — Vollständiger 7B-Run + 3B-Halluzinations-Klassen

Per-Field-Accuracy auf den 12 Hand-Gold-Anzeigen + Schema-Konformitäts-Check auf den restlichen Anzeigen ohne Gold. 3B-vs-7B-vs-Gold per `refnr` joinen, drei eigenständige Halluzinations-Klassen mit konkreten Beispielen identifizieren.

In [2]:
# Absolute Pfade für den Schema-Check des vollen 7B-Runs nutzen
!python /home/jovyan/work/notebooks/LLM-Workshop/llm-workshop/AUFGABEN/annotation/validate.py --validate-jsonl /home/jovyan/work/notebooks/LLM-Workshop/llm-workshop/daten/predictions_full_7b_flat.jsonl


JSONL Schema-Check: predictions_full_7b_flat.jsonl
Geprüfte Zeilen: 165
JSON-Parse-Fails: 0

22 Feld-Verletzungen (sortiert):
     9×  vertragsart: 'nicht_genannt' nicht in Schema
     3×  homeoffice: 'mobiles arbeiten' nicht in Schema
     2×  homeoffice: '100%' nicht in Schema
     2×  erfahrungslevel: 'mehr_jahre' nicht in Schema
     1×  vertragsart: 'befristet' nicht in Schema
     1×  homeoffice: 'flexibles home-office' nicht in Schema
     1×  erfahrungslevel: 'berufserfahrene' nicht in Schema
     1×  homeoffice: 'hybrides' nicht in Schema
     1×  vertragsart: 'direktvermittlung' nicht in Schema
     1×  erfahrungslevel: 'mehrjahrig' nicht in Schema


### Befund zur Pipeline-Stabilität im 7B-Großlauf
* **JSON-Parse-Fails:** 0 (Die strukturelle JSON-Integrität ist bei 7B absolut fehlerfrei).
* **Feld-Verletzungen:** Nur 22 Verletzungen bei insgesamt 990 extrahierten Feldern (165 Anzeigen $\times$ 6 Felder). Die Pipeline ist hochgradig stabil.
* **Auffälligste Verletzungstypen:**
  * **Erfundene Enums:** Das Modell hat bei der Vertragsart vereinzelt `nicht_genannt` halluziniert, obwohl dies im Schema für dieses Feld nicht vorgesehen ist.
  * **Wörtliche Übernahmen (Extractive vs. Abstractive):** Anstatt "Mobiles Arbeiten" oder "hybrides" auf das erlaubte Enum `teilweise` zu mappen, hat das Modell die Begriffe in wenigen Fällen einfach roh aus dem Text kopiert. Gleiches gilt für das Erfahrungslevel (z.B. Erfindung von `berufserfahrene` statt `mid`/`senior`).

In [6]:
# 1. Datensätze laden und IDs bereinigen
gold_df = pd.read_csv(gold_pfad, dtype=str).rename(
    columns={"id": "refnr"}
)
gold_df["refnr"] = gold_df["refnr"].str.strip()


preds_3b = pd.DataFrame(
    [json.loads(z) for z in open(basispfad / "daten" / "predictions_full_3b_flat.jsonl")]
)
preds_7b = pd.DataFrame(
    [json.loads(z) for z in open(basispfad / "daten" / "predictions_full_7b_flat.jsonl")]
)

preds_3b["refnr"] = preds_3b["refnr"].astype(str).str.strip()
preds_7b["refnr"] = preds_7b["refnr"].astype(str).str.strip()



preds_3b = preds_3b.add_suffix('_3b').rename(columns={'refnr_3b': 'refnr'})
preds_7b = preds_7b.add_suffix('_7b').rename(columns={'refnr_7b': 'refnr'})

# 2. 3-Wege-Join über die 12 Hand-Gold-Anzeigen
eval_12 = gold_df.merge(preds_3b, on="refnr").merge(preds_7b, on="refnr")

# 3. Metrik-Rechner für den direkten Modell-Kontrast
def get_acc(df, suffix, feld):
    korrekt = 0
    for _, row in df.iterrows():
        # Gold-Wert sicher auslesen
        w_gold = str(row[feld]).strip().lower() if pd.notna(row[feld]) else ""
        
        # Prediction-Wert sicher auslesen (z.B. 'homeoffice_3b')
        w_pred = row[f"{feld}_{suffix}"]

        if feld == "skills_top3":
            g_set = set([s.strip() for s in w_gold.split("|") if s.strip()])
            p_set = (
                set([str(s).strip().lower() for s in w_pred])
                if isinstance(w_pred, list)
                else set()
            )
            if g_set == p_set:
                korrekt += 1
        elif "gehalt" in feld:
            if w_gold in ["nan", "", "none"]:
                w_gold = None
            else:
                try:
                    w_gold = float(w_gold)
                except ValueError:
                    pass
            
            # None/NaN Handling für Predictions
            if pd.isna(w_pred):
                w_pred = None
                
            if w_gold == w_pred:
                korrekt += 1
        else:
            w_pred_str = str(w_pred).strip().lower() if pd.notna(w_pred) else "none"
            if w_gold == "nan" or w_gold == "":
                w_gold = "none"
                
            if w_gold == w_pred_str:
                korrekt += 1
                
    return round((korrekt / len(df)) * 100, 1)

# Ergebnistabelle aufbauen
kontrast_data = []
for f in [
    "homeoffice",
    "vertragsart",
    "erfahrungslevel",
    "gehalt_min_eur",
    "gehalt_zeitraum",
    "skills_top3",
]:
    kontrast_data.append(
        {
            "Feld": f,
            "Accuracy 3B (%)": get_acc(eval_12, "3b", f),
            "Accuracy 7B (%)": get_acc(eval_12, "7b", f),
        }
    )

print("=== MODELL-KONTRAST ACCURACY AUF DEN 12 HAND-GOLD-ANZEIGEN ===")
pd.DataFrame(kontrast_data)

=== MODELL-KONTRAST ACCURACY AUF DEN 12 HAND-GOLD-ANZEIGEN ===


,Feld,Accuracy 3B (%),Accuracy 7B (%)
0,homeoffice,41.7,41.7
1,vertragsart,58.3,83.3
2,erfahrungslevel,50.0,66.7
3,gehalt_min_eur,75.0,91.7
4,gehalt_zeitraum,75.0,91.7
5,skills_top3,0.0,0.0


In [10]:
# Die Liste aller Felder aus dem Schema
felder = [
    "homeoffice",
    "vertragsart",
    "erfahrungslevel",
    "gehalt_min_eur",
    "gehalt_zeitraum",
    "skills_top3"
]

# Für jedes Feld eine eigene übersichtliche Tabelle generieren
for feld in felder:
    print(f"\n{'='*60}")
    print(f"DETAIL-CHECK: {feld.upper()}")
    print(f"{'='*60}")
    
    # filtern den DataFrame auf die ID und die 3 relevanten Spalten für dieses Feld
    vergleichs_df = eval_12[['refnr', feld, f"{feld}_3b", f"{feld}_7b"]]
    
    # display() sorgt in Jupyter für eine schöne HTML-Tabellendarstellung
    display(vergleichs_df)


DETAIL-CHECK: HOMEOFFICE


,refnr,homeoffice,homeoffice_3b,homeoffice_7b
0,15939-BB-633097-7878-9999-S,teilweise,nicht_genannt,nicht_genannt
1,15939-BB-633095-7878-7490-S,teilweise,nicht_genannt,nicht_genannt
2,15939-BB-633455-7878-6343-S,nicht_genannt,nicht_genannt,nicht_genannt
3,15939-BB-633457-7878-2175-S,nicht_genannt,nicht_genannt,nicht_genannt
4,18777-931781141-S,nicht_genannt,nicht_genannt,nicht_genannt
5,13999-k53401.30280-S,teilweise,nicht_genannt,nicht_genannt
6,15939-BB-632493-7878-9058-S,nicht_genannt,nicht_genannt,nicht_genannt
7,13635-7fbe73ac_JB5131141-S,teilweise,ja,100%
8,20536-lutif851vc-S,teilweise,remote,ja
9,12826-SA0136034_JB5125696-S,teilweise,nicht_genannt,nicht_genannt



DETAIL-CHECK: VERTRAGSART


,refnr,vertragsart,vertragsart_3b,vertragsart_7b
0,15939-BB-633097-7878-9999-S,festanstellung,festanstellung,festanstellung
1,15939-BB-633095-7878-7490-S,festanstellung,senior,festanstellung
2,15939-BB-633455-7878-6343-S,festanstellung,festanstellung,festanstellung
3,15939-BB-633457-7878-2175-S,festanstellung,festanstellung,festanstellung
4,18777-931781141-S,festanstellung,sonstiges,festanstellung
5,13999-k53401.30280-S,festanstellung,festanstellung,direktvermittlung
6,15939-BB-632493-7878-9058-S,praktikum,werkstudent,praktikum
7,13635-7fbe73ac_JB5131141-S,festanstellung,festanstellung,festanstellung
8,20536-lutif851vc-S,sonstiges,festanstellung,festanstellung
9,12826-SA0136034_JB5125696-S,festanstellung,sonstiges,festanstellung



DETAIL-CHECK: ERFAHRUNGSLEVEL


...

,refnr,gehalt_min_eur,gehalt_min_eur_3b,gehalt_min_eur_7b
0,15939-BB-633097-7878-9999-S,NaN,NaN,NaN
1,15939-BB-633095-7878-7490-S,NaN,NaN,NaN
2,15939-BB-633455-7878-6343-S,NaN,NaN,NaN
3,15939-BB-633457-7878-2175-S,NaN,60000.0,NaN
4,18777-931781141-S,NaN,NaN,NaN
5,13999-k53401.30280-S,NaN,NaN,NaN
6,15939-BB-632493-7878-9058-S,NaN,NaN,NaN
7,13635-7fbe73ac_JB5131141-S,NaN,NaN,NaN
8,20536-lutif851vc-S,NaN,50000.0,NaN
9,12826-SA0136034_JB5125696-S,NaN,NaN,NaN



DETAIL-CHECK: GEHALT_ZEITRAUM


,refnr,gehalt_zeitraum,gehalt_zeitraum_3b,gehalt_zeitraum_7b
0,15939-BB-633097-7878-9999-S,NaN,None,None
1,15939-BB-633095-7878-7490-S,NaN,None,None
2,15939-BB-633455-7878-6343-S,NaN,None,None
3,15939-BB-633457-7878-2175-S,NaN,jahr,None
4,18777-931781141-S,NaN,None,None
5,13999-k53401.30280-S,NaN,None,None
6,15939-BB-632493-7878-9058-S,NaN,None,None
7,13635-7fbe73ac_JB5131141-S,NaN,None,None
8,20536-lutif851vc-S,NaN,jahr,None
9,12826-SA0136034_JB5125696-S,NaN,None,None



DETAIL-CHECK: SKILLS_TOP3


,refnr,skills_top3,skills_top3_3b,skills_top3_7b
0,15939-BB-633097-7878-9999-S,ils|opus suite|fmeca,"[data analysis, logistics, technical systems]","[datenanalyse, datenmodellierung, lebenszyklus..."
1,15939-BB-633095-7878-7490-S,opus suite|python,"[data science, simulation, python]","[python, opussuite, reliability_engineering]"
2,15939-BB-633455-7878-6343-S,lakehouse|data lineage,"[data lakehouse architecture, data ingestion p...","[data-lakehouse, data-ingestion-pipelines, dat..."
3,15939-BB-633457-7878-2175-S,mlops|devops,"[machine learning operations, platform enginee...","[machine-learning, devops, docker]"
4,18777-931781141-S,siemens polarion|ibm rational doors|ecss,"[systems engineering, requirements management,...","[siemens polarion, ibm rational doors, systems..."
5,13999-k53401.30280-S,ms excel|erp,"[accounting, controlling, ms_excel]",[ms excel]
6,15939-BB-632493-7878-9058-S,excel|powerpoint,"[excel, powerpoint, englisch]","[power_point, excel]"
7,13635-7fbe73ac_JB5131141-S,python|tensorflow|pytorch,"[machine learning, neuronale netze, deep learn...","[python, tensorflow, scikit-learn]"
8,20536-lutif851vc-S,machine learning|artificial intelligence,"[machine learning, data analysis, technology t...","[machine_learning, artificial_intelligence, da..."
9,12826-SA0136034_JB5125696-S,windows server|linux|ms office 365,"[support, troubleshooting, linux_systeme]","[ms office 365, troubleshooting, linux]"


## Phase 6.2: Drei systematische 3B-Halluzinations-Klassen


### Klasse 1: "Schema-Bruch durch Cross-Field-Leakage"
* **Beschreibung:** Bei komplexeren Texten verliert das 3B-Modell die Zuordnung der JSON-Schlüssel. Es greift Werte auf, die eigentlich zu einem anderen Feld gehören, und bricht damit harte Schema-Vorgaben.
* **Beispiel (ID `15939-BB-633095-7878-7490-S`):**
  * *Gold / 7B sagt:* `vertragsart` = "festanstellung"
  * *3B sagt:* `vertragsart` = "senior"
* **Vermutete Ursache:** Das geringere Aufmerksamkeitsfenster (Attention Mechanism) des 3B-Modells ist überfordert. Das Modell liest "Senior" im Text, behält es im Kurzzeitgedächtnis und feuert es beim falschen JSON-Key ab, ohne zu validieren, ob das Wort überhaupt in die Enum-Liste für Verträge passt.

### Klasse 2: "Gehalts-Erfindung durch assoziatives Zahlen-Matching"
* **Beschreibung:** Das 3B-Modell erträgt keine "leeren" Felder bei Gehältern. Wenn kein Gehalt geboten wird, neigt es dazu, willkürliche Zahlen aus dem Text (z.B. Umsatz des Unternehmens, Postleitzahlen) oder aus seinen Trainingsdaten als Gehalt zu konfabulieren.
* **Beispiel (ID `15939-BB-633457-7878-2175-S` & `20536-lutif851vc-S`):**
  * *Gold / 7B sagt:* `gehalt_min_eur` = `NaN` (keine Angabe)
  * *3B sagt:* `gehalt_min_eur` = 60000.0 bzw. 50000.0
* **Vermutete Ursache:** Dem kleineren Modell fehlt die Nuancierung zwischen einer echten "Gehaltsangabe für den Bewerber" und beliebigen anderen numerischen Werten. Es will das Feld um jeden Preis füllen (Over-Completion).

### Klasse 3: "Kategorie-Verallgemeinerung statt Tool-Extraktion (Buzzword-Washing)"
* **Beschreibung:** Bei der Extraktion der `skills_top3` weicht das 3B-Modell systematisch auf abstrakte Meta-Disziplinen aus, anstatt die konkreten, harten Programmierwerkzeuge zu extrahieren.
* **Beispiel (ID `13635-7fbe73ac_JB5131141-S`):**
  * *Gold sagt:* `python | tensorflow | pytorch` (sowie das 7B-Modell ähnlich mit `scikit-learn`)
  * *3B sagt:* `[machine learning, neuronale netze, deep learning]`
* **Vermutete Ursache:** Das 3B-Modell abstrahiert den Textinhalt zu stark in generische Überbegriffe. Es besitzt nicht das semantische Begriffsverständnis, um "Skills" streng als anwendbare Technologien (Tools/Frameworks) zu definieren.

---

### Implikation für den Produktiveinsatz

**Würden wir das 3B-Modell im Produktivszenario einsetzen?**
Nein, nicht als alleinigen, unüberwachten Annotator. Die Fehler sind zu systematisch und gefährden die Datenintegrität (insbesondere bei frei erfundenen Gehältern oder Schema-Brüchen).

**Wenn ein Einsatz aus Kostengründen zwingend ist, bedarf es folgender Sicherungen:**
1. **Strikter Code-Validator:** Ein Skript (z.B. Pydantic) muss alle Outputs filtern. Wenn ein Wert wie `senior` in der Vertragsart auftaucht, muss der Datensatz sofort blockiert und verworfen (oder an 7B eskaliert) werden.
2. **Eingrenzung auf bestimmte Felder:** Das Modell darf niemals kritische Felder wie Gehalt autonom befüllen. 
3. **Menschliches Stichproben-Review:** Für unstrukturierte Felder (`skills_top3`) muss das 3B-Modell als reines "Vor-Ausfüll-Werkzeug" betrachtet werden, dessen Output zwingend durch einen Menschen (Human-in-the-Loop) bereinigt werden muss.